# Sparse matrices in sPyTial
## Seeing the *layout*, not just the numbers

The canonical Python library for sparse matrices is **[`scipy.sparse`](https://docs.scipy.org/doc/scipy/reference/sparse.html)**
(PyTorch's `torch.sparse` and the PyData [`sparse`](https://sparse.pydata.org) package are the other two you'll meet).

A sparse matrix is *not* really a grid of numbers. It is a small **bundle of parallel arrays** — a
compression scheme (CSR, CSC, COO, …) that stores only the nonzeros. The grid is the *logical* view;
the arrays are the *physical* one. The interesting thing to **see** is the physical bundle, and what an
element-access has to touch across it.

This is exactly the question Rachit Nigam posed:

> My original impetus for trying it out was that I was trying to create visualizations for various sparse
> matrix layouts. … one thing I'd really like to do is visualize what an *access* into any given element in a
> sparse representation would look like — **which related indices do I have to touch across all the data
> structures that store the sparse matrix representation to get this value?**

This notebook builds a small, **layout-aware relationalizer** that answers precisely that, for COO, CSR, and CSC.

## 1. A tiny matrix, three ways to store it

We'll use one 4×4 matrix with five nonzeros throughout.

In [1]:
import numpy as np
import scipy.sparse as sp
import spytial

M = np.array([[0, 9, 0, 0],
              [5, 8, 0, 2],
              [0, 0, 0, 0],
              [0, 0, 7, 0]], dtype=float)

A = sp.csr_matrix(M)
print('dense:')
print(M.astype(int))
print()
print('CSR stores three flat arrays:')
print('  indptr  =', A.indptr,  '  # row r occupies the slice [indptr[r] : indptr[r+1]]')
print('  indices =', A.indices, '  # the COLUMN of each stored nonzero')
print('  data    =', A.data,    '  # the VALUE  of each stored nonzero')

dense:
[[0 9 0 0]
 [5 8 0 2]
 [0 0 0 0]
 [0 0 7 0]]

CSR stores three flat arrays:
  indptr  = [0 1 4 4 5]   # row r occupies the slice [indptr[r] : indptr[r+1]]
  indices = [1 0 1 3 2]   # the COLUMN of each stored nonzero
  data    = [9. 5. 8. 2. 7.]   # the VALUE  of each stored nonzero


## 2. What does `diagram()` do with it today?

Out of the box, sPyTial has no idea what a sparse matrix is, so it falls back to the **generic object
relationalizer**, which introspects attributes with `inspect.getmembers`. The trouble: every attribute of a
numpy array (`.T`, `.real`, `.data`, …) is *itself* an array with the same endless attributes, so the walk
never terminates.

In [2]:
# diagram() builds a data instance first; that's where it blows up.
try:
    spytial.diagram(A)
except RecursionError as e:
    print('💥', type(e).__name__, '—', e)
    print()
    print("The generic relationalizer recursed forever through numpy's array attributes.")
    print('So: yes, sparse matrices need a relationalizer of their own.')

Error during data instance building: Maximum recursion depth exceeded
Object:   (0, 1)	9.0
  (1, 0)	5.0
  (1, 1)	8.0
  (1, 3)	2.0
  (3, 2)	7.0
💥 RecursionError — Maximum recursion depth exceeded

The generic relationalizer recursed forever through numpy's array attributes.
So: yes, sparse matrices need a relationalizer of their own.


## 3. A layout-aware relationalizer

The fix lives in [`sparse_layout.py`](sparse_layout.py) next to this notebook. The idea is to map the
*storage arrays* to atoms and relations, instead of chasing Python attributes:

| atom type | one per | label |
|---|---|---|
| `Ptr`    | entry of `indptr[]`  | the pointer value (a row boundary) |
| `MajIdx` | entry of `indices[]` | the column of a stored nonzero |
| `Val`    | entry of `data[]`    | the value of a stored nonzero |

and relations that encode the layout: `nextPtr`/`nextIdx`/`nextVal` lay each array out left-to-right,
`pair` stacks `data[k]` under `indices[k]`, and **`slice`** draws the key dereference — each row pointer
points at the first column its row owns. A few class-level `@spytial.group` / `@orientation` / `@edgeColor`
decorators turn that graph into a readable picture.

Importing the module registers the relationalizer (priority 100, above all built-ins):

In [3]:
import sparse_layout as sl
from spytial import RelationalizerRegistry

print('top relationalizers by priority:')
for prio, name in RelationalizerRegistry.list_relationalizers()[:3]:
    print(f'  {prio:>3}  {name}')

top relationalizers by priority:
  100  SparseLayoutRelationalizer
   10  PrimitiveRelationalizer
    9  DictRelationalizer


## 4. COO — the uncompressed baseline

The simplest format, **COO** (coordinate list), is just three parallel arrays: `row[]`, `col[]`, `data[]`.
Each column `k` is one nonzero — the triplet `(row[k], col[k], data[k])`. No compression, nothing clever;
it's the mental model everything else compresses.

In [4]:
spytial.diagram(sl.layout(sp.coo_matrix(M)))

## 5. CSR — compress the rows

**CSR** keeps `indices[]` and `data[]` but throws away the per-nonzero `row[]` array, replacing it with a
much shorter `indptr[]` of *row boundaries*. Row `r` lives in `indices[indptr[r] : indptr[r+1]]`. The
vertical **`slice`** edges are that pointer dereference — follow one to see where a row begins in the
indices/data arrays. (Row 2 is empty, so it has no slice edge — its two pointers are equal.)

In [5]:
spytial.diagram(sl.layout(A))   # A is csr

## 6. The access path — *which indices do I touch?*

This is Rachit's question. `sl.access(A, i, j)` draws the same layout, then overlays the exact walk an
element-access performs, colour-coded by step:

- 🟠 **`bound`** — read the two row pointers `indptr[i]` and `indptr[i+1]` (the row's slice).
- ⚪ **`scan`** — walk `indices[]` across that slice, comparing each column to `j`.
- 🟢 **`hit`** — the column that equals `j`, and the `data[]` cell it pairs with — the returned value.

Reading `A[1, 3]`: pointers say row 1 is `data[1:4]`; we scan columns `0, 1, 3`, match `3` at storage
index `k = 3`, and return `data[3] = 2`.

In [6]:
spytial.diagram(sl.access(A, 1, 3))   # present  -> data[3] = 2

Two instructive non-hits — a value that is structurally **absent** (scanned, never matched), and an
access into an **empty row** (the slice is empty, so there is nothing to scan):

In [7]:
spytial.diagram(sl.access(A, 1, 2))   # missing in a non-empty row -> 0, after a full scan

In [8]:
spytial.diagram(sl.access(A, 2, 0))   # empty row -> 0, indptr[2] == indptr[3], no scan

## 7. A different layout touches different indices

The point of having many formats is that they make different accesses cheap. **CSC** is CSR transposed:
`indptr[]` ranges over *columns*, `indices[]` holds *rows*, and an access scans within a column. The same
logical element `A[1, 3]` now lives at a **different** physical index — `data[4]` instead of `data[3]` —
because the nonzeros are stored column-major. Seeing *which* cells light up is the whole game.

In [9]:
B = sp.csc_matrix(M)
print('CSC arrays:  indptr =', B.indptr, ' indices =', B.indices, ' data =', B.data)
spytial.diagram(sl.access(B, 1, 3))   # same element, column-compressed -> data[4] = 2

CSC arrays:  indptr = [0 1 3 4 5]  indices = [1 0 1 3 1]  data = [5. 9. 8. 7. 2.]


## Do we need a better relationalizer for sparse matrices?

**Yes — and it has to be layout-aware.** Two takeaways:

1. The generic relationalizer doesn't just look bad on sparse data; it *fails outright* (infinite recursion
   through numpy arrays). Any numeric/array library needs a purpose-built relationalizer.
2. The *useful* thing to visualize is the **physical layout**, not the logical grid. A sparse matrix is a
   little data-structure-of-arrays, and the questions people actually have — "what does an access cost?",
   "why is CSR fast for row slices and CSC for column slices?" — are questions about that layout. The
   access-path overlay answers *which indices you touch across all the arrays*, directly.

### Where this could go

- **More formats.** BSR (blocked), DIA (diagonal), LIL (list-of-lists) each have their own array bundle and
  their own access story; the same atom/relation recipe extends to them.
- **Other libraries.** `torch.sparse` (COO/CSR tensors) and PyData `sparse` (N-d COO) have the same shape —
  the relationalizer just needs to read their array fields.
- **Ship it.** This is a natural candidate for a `spytial.contrib.sparse` module so `diagram(A)` on any
  scipy/torch sparse matrix *just works*, with `access(A, i, j)` for the path overlay.